# BUSI2066 · Week 9 — Two Categories at Once: Chi-square


### Before you start

Drag in **this notebook** and **`segments.csv`**, and select the kernel.

### Today

Last week both columns were numbers. This week **both are categories** — and most of your survey is
categories. *Which brand do you prefer? Have you bought from them? Which age band are you in?*

> **Is whether a customer subscribes related to which segment they are in?**

Two categories, one table of counts, and one test.

### The rule

> **AI explains. It does not write.**

---
# Part 0 · Set up, and the file

In [ ]:
%pip install -q pandas seaborn scipy statsmodels

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('segments.csv')

print(df.shape)
df.head()

**300 customers, 7 columns**, from a marketing database that groups customers into segments.

| Column | Type | Values |
|---|---|---|
| `age` · `income` · `kids` | numbers | |
| `gender` | nominal | Female · Male |
| `ownHome` | nominal | ownYes · ownNo |
| `subscribe` | nominal | subYes · subNo — **subscribes to the company's service** |
| `Segment` | nominal | Suburb mix · Travelers · Moving up · Urban hip |

⚠️ **Simulated.** Round group sizes (100 · 80 · 70 · 50) and an income column with a **negative
minimum** are the giveaways — real incomes are not negative. Use it for the method, not for the
business conclusion.

In [ ]:
df.describe(include='all')

`include='all'` makes `describe()` report the text columns too: `unique`, `top` and `freq` rows.
Read the `top` row — the most common value of each category. `subNo` with freq 260 means **only 40
of 300 subscribe**. That number matters all day.

---
# Part 1 · Describe the difference

Three research questions:

| RQ | Question | Type |
|---|---|---|
| **RQ1** | Are the four segments the same size? | Descriptive |
| **RQ2** | Is subscribing related to segment? | Relational |
| **RQ3** | Is subscribing related to gender? | Relational |

## 1.1 One category — counts and proportions

You know this from week 5.

In [ ]:
print(df['Segment'].value_counts())
print()
print(df['Segment'].value_counts(normalize=True).round(3))

100 · 80 · 70 · 50. Not equal — but is that *sampling variability*, or are the segments really
different sizes? That is RQ1, and it is Part 3.

## 1.2 Two categories — the crosstab

A **two-way frequency table** counts how many customers fall in every combination of two categories.
Pandas calls it a crosstab.

In [ ]:
table2 = pd.crosstab(df['Segment'], df['subscribe'])
table2

Rows are segments, columns are subscribe/not, cells are counts. **This table is the whole of today's
analysis** — everything else is either drawing it or testing it.

Read it: Moving up has 14 subscribers out of 70; Suburb mix has 6 out of 100. The raw counts are
misleading because the groups are different sizes. You want **rates**.

## 1.3 Two-way proportions — the rate per row

In [ ]:
pd.crosstab(df['Segment'], df['subscribe'], normalize='index').round(3)

`normalize='index'` divides each **row** by its total, so every row sums to 1. Now you can compare:

| Segment | Subscribe rate |
|---|---|
| Moving up | **20.0%** |
| Urban hip | **20.0%** |
| Travelers | 12.5% |
| Suburb mix | **6.0%** |

Moving up and Urban hip subscribe at more than three times the rate of Suburb mix. That is RQ2,
described. Whether it is *real* is Part 3.

⚠️ **`normalize='index'` is rows; `normalize='columns'` is columns; `normalize=True` is the whole
table.** For "the subscribe rate *within each segment*", it is rows. Getting this wrong produces a
table that looks fine and means something else.

## 1.4 Gender — the same two tables

In [ ]:
table3 = pd.crosstab(df['gender'], df['subscribe'])
print(table3)
print()
print(pd.crosstab(df['gender'], df['subscribe'], normalize='index').round(3))

**13.4% of women subscribe, 13.3% of men.** Those are as close as two numbers get. Remember this one
— Part 3 tests it.

## 1.5 The marginal sums

The row and column totals of a two-way table are its **marginal sums** — they are written in the
margins. Part 3 needs them.

In [ ]:
pd.crosstab(df['gender'], df['subscribe'], margins=True)

📖 `margins=True` adds a row and a column called `All`: the marginal sums. Read-level.

157 women and 143 men; 260 non-subscribers and 40 subscribers; 300 in all. Keep these numbers for §3.2.

---
# Part 2 · Draw the difference

## 2.1 One category — one bar per group

First, how many customers are in each segment? `y=` instead of `x=` gives horizontal bars,
which leave room for long labels.

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, y='Segment', color='lightblue')
plt.xlabel('Number of customers')
plt.title('Customers per segment')
plt.show()

Suburb mix is the biggest, Urban hip the smallest. That is RQ1 drawn. `color=` gives every bar one colour.

## 2.2 Two categories — counts, two ways

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Segment', hue='subscribe', palette='Set2')
plt.title('Subscribers and non-subscribers by segment')
plt.show()

`hue='subscribe'` splits every bar in two — the grouped bar chart. It shows **counts**, so Suburb
mix looks like the big segment (it is) and its 6 subscribers are hard to see.

## 2.3 Two categories — rates

For the **rate**, you want each segment's bar to be the same height and split by proportion.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x='Segment', hue='subscribe', multiple='fill', shrink=0.8, palette='Set2')
plt.ylabel('Proportion')
plt.title('Subscribe rate by segment')
plt.show()

📖 `histplot` with `multiple='fill'` is the **filled bar chart** — every bar reaches 1, split by
proportion. `shrink=0.8` puts gaps between the bars. This is read-level: copy it when you want rates.

Now the 20% / 20% / 12.5% / 6% pattern is visible, and Suburb mix's size no longer distorts it.

**Counts when you want to show how many. Filled bars when you want to show the rate.** A report
needs both.

## 2.4 Gender — the same two charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.countplot(data=df, x='gender', hue='subscribe', palette='Set2', ax=axes[0])
axes[0].set_title('Subscribers by gender — counts')

sns.histplot(data=df, x='gender', hue='subscribe', multiple='fill', shrink=0.8, palette='Set2', ax=axes[1])
axes[1].set_ylabel('Proportion')
axes[1].set_title('Subscribe rate by gender')

plt.show()

Left: more women than men in both bars, because there are more women. Right: the subscribe part is the
same height for both — 13.4% and 13.3%. **The filled chart shows there is almost no difference.**

---
# Part 3 · Is it real? — the chi-square test

A chi-square test compares the counts you **observed** (O) with the counts you would **expect if H₀
were true** (E), and asks whether the gap is bigger than sampling variability would produce.

$$\chi^2 = \sum \frac{(O - E)^2}{E}$$

In words: for **every cell**, take the gap O − E, **square** it (so minus and plus gaps do not cancel),
and **divide by E** (a gap of 5 matters more when only 10 were expected). Then **add up** over all cells.

**Where the p-value comes from.** If H₀ is true, χ² follows a known curve, the **χ² distribution** — its
sampling distribution (week 6). It is never below 0 and is skewed to the right. Its shape depends on
the **degrees of freedom (df)**:

- one category with *k* groups: df = *k* − 1
- two categories with *k* and *n* groups: df = (*k* − 1) × (*n* − 1)

The p-value is the area of that curve to the right of our χ²: *if H₀ were true, how often would a gap
this big or bigger turn up?* Under 5%: reject H₀. The slides show the curve.

## 3.1 One category — are the segments the same size?

| | |
|---|---|
| **H₀** | The four segments have equal numbers of customers |
| **H₁** | They do not |

If H₀ were true, each segment would have 300 ÷ 4 = **75** customers. We observed 100 · 80 · 70 · 50.

In [ ]:
from scipy.stats import chisquare

counts = df['Segment'].value_counts()

result = chisquare(f_obs=counts)          # expected = equal shares, by default

print("chi-square:", result.statistic)
print("p:", result.pvalue)

**χ² = 17.3, p = 0.0006.** Reject H₀ — the segments are not the same size in the population.

`f_obs` is the observed counts. Leave `f_exp` out and the test assumes equal expected counts, which
is what H₀ says here. You can supply `f_exp=[...]` for any other expectation.

### The test is sensitive to sample size

Same proportions, a tenth of the data:

In [ ]:
result = chisquare(f_obs=counts / 10)      # 10 · 8 · 7 · 5

print("chi-square:", result.statistic)
print("p:", result.pvalue)

Identical proportions — 33%, 27%, 23%, 17% — and **p = 0.63**. Not significant.

The difference between the groups did not change. The **evidence** for it did. Thirty customers
cannot establish what three hundred can. This is week 6's √n, again, and it is why your project
brief sets a minimum sample size.

## 3.2 Two categories — is subscribing related to gender?

| | |
|---|---|
| **H₀** | Subscribing is **not** related to gender — they are independent |
| **H₁** | They are related |

For a two-way table the test is `chi2_contingency`, and you hand it the crosstab.

In [ ]:
from scipy.stats import chi2_contingency

result = chi2_contingency(table3)

print("chi-square:", result.statistic)
print("p:", result.pvalue)
print("degrees of freedom:", result.dof)
print()
print("expected counts if H0 were true:")
print(result.expected_freq.round(1))

**χ² = 0.0, p = 1.0.** You will never see a cleaner "no". We fail to reject H₀.

Look at `expected_freq`. Each expected count is **row marginal sum × column marginal sum ÷ total**
(§1.5): if gender and subscribing were unrelated, 157 women × 40 / 300 = **20.9** women would
subscribe. We observed 21. The test does this for every cell, then applies the formula above.

**Why exactly 0?** Every gap here is under 0.1. By the formula, χ² would be 0.0005 — tiny, but not 0.
For a 2 × 2 table, `chi2_contingency` makes a small safety correction by default (**Yates'
correction**): it moves each observed count up to 0.5 closer to its expected count before squaring.
Our gaps are smaller than 0.5, so they become zero. Switch the correction off to see the formula's value:

In [ ]:
result = chi2_contingency(table3, correction=False)

print("chi-square:", result.statistic)
print("p:", result.pvalue)

χ² = 0.0005, p = 0.98. **Same conclusion.** 📖 `correction=False` is read-level; keep the default in
your own work. It only matters for 2 × 2 tables.

**Conclusion for RQ3: no evidence that subscribing is related to gender.** A finding.

## 3.3 Is subscribing related to segment?

In [ ]:
result = chi2_contingency(table2)

print("chi-square:", result.statistic)
print("p:", result.pvalue)
print("degrees of freedom:", result.dof)

**χ² = 9.3, p = 0.025.** Reject H₀ — subscribing **is** related to segment. The 20% / 6% gap you
described in §1.3 is bigger than chance would produce.

What the test does **not** say: *which* segments differ, or *why*. It says the table is not what
independence would give. The rates table from §1.3 is where the story is; the test is what lets you
tell it.

`dof` — degrees of freedom — is (rows − 1) × (columns − 1) = 3 × 1 = 3. It tells the test
which χ² curve to read the p-value from.

---
# Part 4 · One proportion — the binomial test

Sometimes the question is not "are two things related?" but **"is this rate what we claimed?"**

> The company says 10% of customers subscribe. Our sample has 40 of 300 — 13.3%. Is the claim wrong?

| | |
|---|---|
| **H₀** | The population subscribe rate is 0.10 |
| **H₁** | It is not 0.10 |

In [ ]:
from scipy.stats import binomtest

result = binomtest(k=40, n=300, p=0.10, alternative='two-sided')

print("observed proportion:", result.statistic)
print("p:", result.pvalue)
print("95% CI:", result.proportion_ci(confidence_level=0.95))

| Argument | Means |
|---|---|
| `k=40` | successes observed |
| `n=300` | trials |
| `p=0.10` | **the rate in H₀** |

**p = 0.067.** Above 0.05 — we fail to reject H₀. 13.3% is not far enough from 10% for 300 customers
to rule out that the true rate is 10%. And the interval **9.7% → 17.7% contains 0.10**.

This is the **one-sample t-test's cousin**: week 6 tested a mean against a claimed value; this tests
a proportion against a claimed value. Same logic, same reading.

## 4.1 A second example — are half the customers women?

157 of the 300 customers are female.

| | |
|---|---|
| **H₀** | The population proportion of women is 0.5 |
| **H₁** | It is not 0.5 |

In [ ]:
result = binomtest(k=157, n=300, p=0.5, alternative='two-sided')

print("observed proportion:", result.statistic)
print("p:", result.pvalue)
print("95% CI:", result.proportion_ci(confidence_level=0.95))

**52.3% women, p = 0.45.** Fail to reject H₀. The interval, **46.5% → 58.1%**, contains 0.5: no evidence
that the population is not half women.

---
# 🛠 Exercise · Home ownership

Complete **all nine questions in class**.
There are **no take home questions**. These class exercises carry **no marks**.

No answers are printed here: run the code and read your own output.

**1 · Describe.** Two-way table of `Segment` against `ownHome`, then the same as rates per segment.

Which segment has the highest home-ownership rate, and which the lowest?

_(your answer)_

In [ ]:
# Your code here

**2 · Draw.** A filled bar chart of ownership rate by segment, with labels and a title.

In [ ]:
# Your code here

**3 · Test.** Is home ownership related to segment? State H₀ and H₁, run the test, conclude.

- **H₀:** _(your answer)_
- **H₁:** _(your answer)_
- **Conclusion:** _(your answer)_

In [ ]:
# Your code here

**4 · Expected counts.** From your result in question 3, how many Travelers would you expect to own a
home **if H₀ were true**? Compare it with the observed count.

_(your answer)_

In [ ]:
# Your code here

**5 · One category.** Is the split between owners and non-owners 50/50 in the population?
Use a chi-square test **and** a binomial test, and check they agree.

_(your answer)_

In [ ]:
# Your code here

**6 · Gender and ownership.** Test whether home ownership is related to gender.

Write the one-sentence conclusion for a report.

_(your answer)_

In [ ]:
# Your code here

**7 · No code.** In question 3 the test said ownership is related to segment. The **Travelers**
segment is the oldest (mean age 58) and has no children.

Give one reason the segment–ownership relationship might **not** be about segment at all. What
week-4 word describes it?

_(your answer)_

**8 · Put it together.** The marketing team wants more subscribers. Someone says: *"Home owners have
more money. Let's send the subscription offer only to home owners."*

Do the four steps in order. Each step uses what the step before it told you.

**a · Predict, then describe.** Question 3 found that owning a home is related to segment. §3.3 found
that subscribing is related to segment. So do you expect owning a home to be related to subscribing?
Write your guess **first**. Then make the two-way table of `ownHome` against `subscribe`, and the same
as rates per row.

- **My guess:** _(your answer)_
- **What the rates show:** _(your answer)_

**b · Draw.** A filled bar chart of the subscribe rate for owners and non-owners, with labels and a title.

**c · Test.** Is subscribing related to home ownership? Report χ², p and the degrees of freedom.

- **H₀:** _(your answer)_
- **H₁:** _(your answer)_
- **Conclusion:** _(your answer)_

**d · Decide.** Write two or three sentences for the marketing team. Should the offer go only to home
owners? If not, which column from today would you use to choose who gets it, and why? Quote the
rates and p-values you rely on. Was your guess in (a) right?

_(your answer)_

In [ ]:
# Your code here

**9 · Check expected counts.** The test in §3.3 said subscribing is related to segment, p = 0.025. But it was run on 300 rows, and
the smallest cell in the table — Suburb mix subscribers — holds **6 people**.

Ask your AI: *"What is the minimum expected cell count for a chi-square test to be reliable, and
what should I do if a cell is below it?"*

Then check: print `result.expected_freq` for `table2`. Is any cell below the threshold it named?
Would you report this p-value without a caveat?

_(your answer)_

In [ ]:
# Your code here

---
# ✅ Done — hand it in

**Complete all nine questions before you submit. No cell is designed to fail.**

1. **Run All**, no red. 2. Save, **commit and sync**. 3. On Moodle: **this notebook**. 4. **Stop your Codespace**.

## What you can now do

- [ ] build a two-way table with `pd.crosstab`, and turn it into **rates per row** with `normalize='index'`
- [ ] draw one bar per group with `countplot`, grouped bars for counts and **filled bars for rates**, and say when each is right
- [ ] run a one-way chi-square on one category, and a two-way chi-square on a crosstab
- [ ] read `expected_freq`, and compute one expected count from the **marginal sums**
- [ ] write the χ² formula in words, and say where its p-value comes from
- [ ] explain why the same proportions with a tenth of the data stop being significant
- [ ] run a binomial test on a single rate against a claimed value
- [ ] say what a significant chi-square does **not** tell you — which groups, or why

## Coming up

**Your final survey is due at the end of this week.** **Next week:** a number across groups — the
two-sample t-test and ANOVA.